# 📊 Sales Data Analysis
**Objective:** Analyze sales dataset to find total revenue, best-selling products, regional performance, and customer insights.

---

## Day 1: Setup & Load Data

In [1]:
# Import required libraries
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Load the CSV file into a DataFrame
df = pd.read_csv('sales_data.csv')

print('✅ Data loaded successfully!')
print(f'   Rows: {df.shape[0]} | Columns: {df.shape[1]}')

✅ Data loaded successfully!
   Rows: 100 | Columns: 7


## Day 2: Explore Data

In [2]:
# Preview the first 5 rows
print('=== First 5 Rows ===')
df.head()

=== First 5 Rows ===


,Date,Product,Quantity,Price,Customer_ID,Region,Total_Sales
0,2024-01-01,Phone,7,37300,CUST001,East,261100
1,2024-01-02,Headphones,4,15406,CUST002,North,61624
2,2024-01-03,Phone,2,21746,CUST003,West,43492
3,2024-01-04,Headphones,1,30895,CUST004,East,30895
4,2024-01-05,Laptop,8,39835,CUST005,North,318680


In [3]:
# Check data types and basic info
print('=== Data Types ===')
print(df.dtypes)
print()
print('=== Unique Products ===')
print(df['Product'].unique())
print()
print('=== Unique Regions ===')
print(df['Region'].unique())

=== Data Types ===
Date           object
Product        object
Quantity        int64
Price           int64
Customer_ID    object
Region         object
Total_Sales     int64
dtype: object

=== Unique Products ===
['Phone' 'Headphones' 'Laptop' 'Tablet' 'Monitor']

=== Unique Regions ===
['East' 'North' 'West' 'South']


In [4]:
# Statistical summary of numerical columns
print('=== Statistical Summary ===')
df.describe().round(2)

=== Statistical Summary ===


,Quantity,Price,Total_Sales
count,100.00,100.00,100.00
mean,4.78,25808.51,123650.48
std,2.59,13917.63,100161.09
min,1.00,1308.00,6540.00
25%,2.75,14965.25,39517.50
50%,5.00,24192.00,97955.50
75%,7.00,38682.25,175792.50
max,9.00,49930.00,373932.00


## Day 3: Clean Data

In [5]:
# Check for missing values in each column
print('=== Missing Values ===')
missing = df.isnull().sum()
print(missing)
print(f'\nTotal missing values: {missing.sum()}')

=== Missing Values ===
Date           0
Product        0
Quantity       0
Price          0
Customer_ID    0
Region         0
Total_Sales    0
dtype: int64

Total missing values: 0


In [6]:
# Check for duplicate rows
duplicates = df.duplicated().sum()
print(f'Duplicate rows found: {duplicates}')

# Drop duplicates if any exist
df = df.drop_duplicates()
print(f'Rows after deduplication: {len(df)}')

# Convert Date column to datetime for time-series analysis
df['Date'] = pd.to_datetime(df['Date'])
df['Month'] = df['Date'].dt.to_period('M')
df['Month_Name'] = df['Date'].dt.strftime('%B %Y')

print('\n✅ Data is clean and ready for analysis!')

Duplicate rows found: 0
Rows after deduplication: 100

✅ Data is clean and ready for analysis!


## Day 4: Analyze Sales

In [7]:
# --- METRIC 1: Overall Revenue Metrics ---
total_revenue    = df['Total_Sales'].sum()
total_orders     = len(df)
avg_order_value  = df['Total_Sales'].mean()
total_units_sold = df['Quantity'].sum()

print('=' * 45)
print('         OVERALL REVENUE METRICS')
print('=' * 45)
print(f'  Total Revenue    : ₹{total_revenue:>15,.2f}')
print(f'  Total Orders     : {total_orders:>18,}')
print(f'  Avg Order Value  : ₹{avg_order_value:>15,.2f}')
print(f'  Total Units Sold : {total_units_sold:>18,}')
print('=' * 45)

         OVERALL REVENUE METRICS
  Total Revenue    : ₹  12,365,048.00
  Total Orders     :                100
  Avg Order Value  : ₹     123,650.48
  Total Units Sold :                478


In [8]:
# --- METRIC 2: Product Performance ---
product_revenue = df.groupby('Product').agg(
    Revenue=('Total_Sales', 'sum'),
    Units_Sold=('Quantity', 'sum'),
    Orders=('Total_Sales', 'count'),
    Avg_Price=('Price', 'mean')
).sort_values('Revenue', ascending=False)

# Add revenue share percentage
product_revenue['Revenue_Share_%'] = (product_revenue['Revenue'] / total_revenue * 100).round(1)
product_revenue['Revenue'] = product_revenue['Revenue'].map('₹{:,.0f}'.format)
product_revenue['Avg_Price'] = product_revenue['Avg_Price'].map('₹{:,.0f}'.format)

print('=== METRIC 2: Product Performance ===')
product_revenue

=== METRIC 2: Product Performance ===


,Revenue,Units_Sold,Orders,Avg_Price,Revenue_Share_%
Product,,,,,
Laptop,"₹3,889,210",136,24,"₹27,652",31.5
Tablet,"₹2,884,340",127,26,"₹24,177",23.3
Phone,"₹2,859,394",101,20,"₹27,379",23.1
Headphones,"₹1,384,033",48,15,"₹28,692",11.2
Monitor,"₹1,348,071",66,15,"₹20,710",10.9


In [9]:
# --- METRIC 3: Regional Performance ---
region_stats = df.groupby('Region').agg(
    Revenue=('Total_Sales', 'sum'),
    Orders=('Total_Sales', 'count'),
    Avg_Sale=('Total_Sales', 'mean')
).sort_values('Revenue', ascending=False)

region_stats['Revenue_Share_%'] = (region_stats['Revenue'] / total_revenue * 100).round(1)
region_stats['Revenue']  = region_stats['Revenue'].map('₹{:,.0f}'.format)
region_stats['Avg_Sale'] = region_stats['Avg_Sale'].map('₹{:,.0f}'.format)

print('=== METRIC 3: Regional Performance ===')
region_stats

=== METRIC 3: Regional Performance ===


,Revenue,Orders,Avg_Sale,Revenue_Share_%
Region,,,,
North,"₹3,983,635",28,"₹142,273",32.2
South,"₹3,737,852",27,"₹138,439",30.2
East,"₹2,519,639",19,"₹132,613",20.4
West,"₹2,123,922",26,"₹81,689",17.2


In [10]:
# --- METRIC 4: Monthly Revenue Trend ---
monthly_revenue = df.groupby('Month')['Total_Sales'].sum().reset_index()
monthly_revenue.columns = ['Month', 'Revenue']
monthly_revenue['Revenue_Formatted'] = monthly_revenue['Revenue'].map('₹{:,.0f}'.format)

print('=== METRIC 4: Monthly Revenue Trend ===')
print(monthly_revenue[['Month', 'Revenue_Formatted']].to_string(index=False))

=== METRIC 4: Monthly Revenue Trend ===
  Month Revenue_Formatted
2024-01        ₹4,120,524
2024-02        ₹2,656,050
2024-03        ₹4,485,006
2024-04        ₹1,103,468


In [11]:
# --- METRIC 5: Top 5 Customers ---
top_customers = df.groupby('Customer_ID')['Total_Sales'].sum().sort_values(ascending=False).head(5)
top_customers_pct = (top_customers / total_revenue * 100).round(1)

print('=== METRIC 5: Top 5 Customers by Revenue ===')
for rank, (cid, rev) in enumerate(top_customers.items(), 1):
    pct = top_customers_pct[cid]
    print(f'  #{rank}  {cid}  ₹{rev:>10,.0f}  ({pct}% of total)')

=== METRIC 5: Top 5 Customers by Revenue ===
  #1  CUST016  ₹   373,932  (3.0% of total)
  #2  CUST007  ₹   363,870  (2.9% of total)
  #3  CUST083  ₹   350,888  (2.8% of total)
  #4  CUST073  ₹   349,510  (2.8% of total)
  #5  CUST020  ₹   333,992  (2.7% of total)


## Day 5: Key Insights Summary

In [12]:
# Derive key insights programmatically
raw_product_rev = df.groupby('Product')['Total_Sales'].sum().sort_values(ascending=False)
raw_region_rev  = df.groupby('Region')['Total_Sales'].sum().sort_values(ascending=False)

best_product      = raw_product_rev.index[0]
best_product_rev  = raw_product_rev.iloc[0]
best_region       = raw_region_rev.index[0]
best_region_rev   = raw_region_rev.iloc[0]
top_2_products_pct = (raw_product_rev.iloc[:2].sum() / total_revenue * 100)
top_customer_rev  = df.groupby('Customer_ID')['Total_Sales'].sum().max()

print()
print('┌─────────────────────────────────────────────────┐')
print('│              KEY BUSINESS INSIGHTS              │')
print('├─────────────────────────────────────────────────┤')
print(f'│  💰 Total Revenue   : ₹{total_revenue:,.0f}           │')
print(f'│  🏆 Best Product    : {best_product} (₹{best_product_rev:,.0f})  │')
print(f'│  🌍 Top Region      : {best_region} (₹{best_region_rev:,.0f})    │')
print(f'│  📦 Top 2 Products  : {top_2_products_pct:.1f}% of all revenue      │')
print(f'│  ⭐ Best Customer   : ₹{top_customer_rev:,.0f} (single cust.) │')
print('└─────────────────────────────────────────────────┘')


┌─────────────────────────────────────────────────┐
│              KEY BUSINESS INSIGHTS              │
├─────────────────────────────────────────────────┤
│  💰 Total Revenue   : ₹12,365,048           │
│  🏆 Best Product    : Laptop (₹3,889,210)  │
│  🌍 Top Region      : North (₹3,983,635)    │
│  📦 Top 2 Products  : 54.8% of all revenue      │
│  ⭐ Best Customer   : ₹373,932 (single cust.) │
└─────────────────────────────────────────────────┘
